# Minesweeper with Tabular Reinforcement Learning

This notebook is a small in-class exercise for experimenting with tabular RL on tiny Minesweeper boards.

The code below gives you:
- a tiny single-agent environment
- one complete random baseline
- one tabular agent skeleton that currently does not learn
- visualization helpers for comparing several agents

This version starts at `3x3` with `2` mines. That is still small enough for a tabular method, but large enough that the state representation starts to matter.

## What the notebook gives you

You do not need to build the game mechanics from scratch.

Your main job is to decide how the agent should:
- represent values in a tabular form
- choose actions from those values
- update the table from experience

The notebook is intentionally light on RL guidance. The creative part is yours.

In [ ]:
import random
from collections import defaultdict
from dataclasses import dataclass

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from mine_board import MineBoard

## State and action representation

The agent only sees the visible board, not the hidden mine layout.

State representation used in this notebook:
- flatten the visible board row by row
- keep each visible symbol as it is: `?`, `' '`, `'1'`, `'2'`, ...
- use the resulting tuple as the state key

This is simple, but the choice matters. In tabular RL, a larger state description means a larger table.

For a `3x3` board with `2` mines, a rough upper bound is:
- each visible cell is one of `?`, `' '`, `'1'`, `'2'`
- so the number of visible states is at most `4^9 = 262,144`
- with `9` actions, that gives at most `262,144 x 9 = 2,359,296` state-action entries

That upper bound is loose, because many boards are impossible and many states are unreachable. Still, it is large enough that representing the state efficiently is worth thinking about.

Example:

```python
visible_board = [
    ['?', '?', '?'],
    ['1', ' ', '?'],
    ['1', '1', '?'],
]

state = ('?', '?', '?', '1', ' ', '?', '1', '1', '?')
```

Action representation used in this notebook:
- each action is one integer
- actions follow row-major order

For a `3x3` board, the actions are `0, 1, ..., 8`.

In [ ]:
def action_to_rc(action, size):
    return divmod(action, size)


def rc_to_action(row, col, size):
    return row * size + col


def flatten_visible_board(board):
    return tuple(cell for row in board for cell in row)


def unflatten_state(state, size):
    return [list(state[row * size:(row + 1) * size]) for row in range(size)]


def moving_average(values, window):
    if not values:
        return []
    averages = []
    for index in range(len(values)):
        start = max(0, index - window + 1)
        chunk = values[start:index + 1]
        averages.append(sum(chunk) / len(chunk))
    return averages


def plot_board(state, size, title='', exploded_action=None, ax=None):
    board = unflatten_state(state, size)
    created_ax = ax is None
    if created_ax:
        _, ax = plt.subplots(figsize=(2.8, 2.8))

    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.axis('off')

    for row in range(size):
        for col in range(size):
            action = rc_to_action(row, col, size)
            cell = board[row][col]
            facecolor = '#d9dde3'
            text = cell
            text_color = '#222222'

            if cell == '?':
                facecolor = '#7f8fa6'
                text_color = 'white'
            elif cell == ' ':
                facecolor = '#f4f5f7'
                text = ''

            if exploded_action == action:
                facecolor = '#d9534f'
                text = 'X'
                text_color = 'white'

            rect = plt.Rectangle((col, row), 1, 1, facecolor=facecolor, edgecolor='black', linewidth=1.5)
            ax.add_patch(rect)
            ax.text(col + 0.5, row + 0.5, text, ha='center', va='center', fontsize=15, color=text_color)

    if title:
        ax.set_title(title, fontsize=10)

    if created_ax:
        plt.show()

## Tiny environment

This wrapper turns `MineBoard` into a small RL environment.

Design choices here:
- `safe_first_move=True` by default
- the environment returns a flattened visible-board state
- the action is one integer cell index
- reward values are simple and easy to change

In [ ]:
@dataclass
class StepInfo:
    won: bool
    exploded: bool
    exploded_action: int | None
    steps: int


class MinesweeperEnv:
    def __init__(
        self,
        size=3,
        num_mines=2,
        safe_first_move=True,
        safe_reward=1.0,
        mine_reward=-5.0,
        win_reward=10.0,
        step_penalty=-0.1,
        seed=0,
    ):
        self.size = size
        self.num_mines = num_mines
        self.safe_first_move = safe_first_move
        self.safe_reward = safe_reward
        self.mine_reward = mine_reward
        self.win_reward = win_reward
        self.step_penalty = step_penalty
        self.seed = seed
        self._rng = random.Random(seed)
        self._board = None
        self._done = False
        self._won = False
        self._steps = 0
        self._exploded_action = None

    def _empty_visible_board(self):
        return tuple(tuple(MineBoard.HIDDEN for _ in range(self.size)) for _ in range(self.size))

    def _sample_board(self, forbidden_action=None):
        forbidden_rc = None
        if forbidden_action is not None:
            forbidden_rc = action_to_rc(forbidden_action, self.size)

        while True:
            board_rng = random.Random(self._rng.randrange(10**9))
            board = MineBoard(size=self.size, num_mines=self.num_mines, rng=board_rng)
            if forbidden_rc is None or not board.has_mine(*forbidden_rc):
                return board

    def _visible_board(self):
        if self._board is None:
            return self._empty_visible_board()
        return tuple(tuple(row) for row in self._board.board())

    def _state(self):
        return flatten_visible_board(self._visible_board())

    def reset(self):
        self._board = None if self.safe_first_move else self._sample_board()
        self._done = False
        self._won = False
        self._steps = 0
        self._exploded_action = None
        return self._state()

    def legal_actions(self):
        if self._done:
            return []
        if self._board is None:
            return list(range(self.size * self.size))

        actions = []
        for row in range(self.size):
            for col in range(self.size):
                if self._board.is_hidden(row, col):
                    actions.append(rc_to_action(row, col, self.size))
        return actions

    def step(self, action):
        if self._done:
            raise RuntimeError('episode already finished')
        if action not in self.legal_actions():
            raise ValueError(f'illegal action: {action}')

        if self._board is None:
            forbidden = action if self.safe_first_move else None
            self._board = self._sample_board(forbidden_action=forbidden)

        row, col = action_to_rc(action, self.size)
        self._steps += 1
        reward = self.step_penalty

        if self._board.has_mine(row, col):
            self._done = True
            self._won = False
            self._exploded_action = action
            reward += self.mine_reward
        else:
            self._board.perform_action(row, col)
            reward += self.safe_reward
            if self._board.is_solved():
                self._done = True
                self._won = True
                reward += self.win_reward

        next_state = self._state()
        info = StepInfo(
            won=self._won,
            exploded=self._done and not self._won,
            exploded_action=self._exploded_action,
            steps=self._steps,
        )
        return next_state, reward, self._done, info

    def render(self, state=None, title=''):
        if state is None:
            state = self._state()
        plot_board(state, size=self.size, title=title, exploded_action=self._exploded_action)

In [ ]:
env = MinesweeperEnv(size=3, num_mines=2, safe_first_move=True, seed=7)
state = env.reset()
print('Initial state:', state)
print('Legal actions:', env.legal_actions())
env.render(title='Initial visible board')

## Agents

The notebook gives you one complete baseline and one incomplete tabular agent.

The tabular agent below currently behaves almost like a random policy:
- it stores a table in `self.q`
- it does not yet use that table in a meaningful way
- it does not yet update the table

Your job is to change that.

In [ ]:
class RandomAgent:
    def __init__(self, seed=0, name='random'):
        self.rng = random.Random(seed)
        self.name = name

    def begin_episode(self):
        pass

    def select_action(self, state, legal_actions):
        return self.rng.choice(list(legal_actions))

    def observe(self, state, action, reward, next_state, done, next_legal_actions):
        pass

    def end_episode(self):
        pass


class TabularAgent:
    def __init__(self, alpha=0.1, gamma=0.95, epsilon=0.1, seed=0, name='tabular_stub'):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.rng = random.Random(seed)
        self.name = name
        self.q = defaultdict(float)

    def begin_episode(self):
        pass

    def select_action(self, state, legal_actions):
        # TODO:
        # Replace this with something like epsilon-greedy action selection.
        # Right now the agent ignores self.q and behaves like a random policy.
        return self.rng.choice(list(legal_actions))

    def observe(self, state, action, reward, next_state, done, next_legal_actions):
        # TODO:
        # Update self.q[(state, action)] here.
        # You can try Q-learning, SARSA, Monte Carlo updates, or another tabular idea.
        pass

    def end_episode(self):
        pass

    def num_visited_states(self):
        return len({state for state, _ in self.q.keys()})

## Episode loop, training, and visualization

This part is ready to use. Add more agents by putting more constructors into `agent_builders` later in the notebook.

In [ ]:
def run_episode(env, agent, train=True, max_steps=100, record=False):
    state = env.reset()
    legal_actions = env.legal_actions()
    agent.begin_episode()

    total_reward = 0.0
    frames = [{'state': state, 'title': 'start'}] if record else None

    for step_index in range(max_steps):
        action = agent.select_action(state, legal_actions)
        next_state, reward, done, info = env.step(action)
        next_legal_actions = env.legal_actions() if not done else []

        if train:
            agent.observe(state, action, reward, next_state, done, next_legal_actions)

        total_reward += reward

        if record:
            outcome = 'safe'
            if info.exploded:
                outcome = 'mine'
            elif info.won:
                outcome = 'win'
            frames.append(
                {
                    'state': next_state,
                    'title': f't={step_index + 1}, a={action}, r={reward:.1f}, {outcome}',
                    'exploded_action': info.exploded_action,
                }
            )

        state = next_state
        legal_actions = next_legal_actions

        if done:
            break

    agent.end_episode()
    return {
        'total_reward': total_reward,
        'won': info.won,
        'steps': info.steps,
        'rollout': frames,
    }


def train_agent(agent, env_config, train_episodes=200, eval_episodes=50, base_seed=0):
    reward_history = []
    win_history = []
    step_history = []

    for episode_index in range(train_episodes):
        env = MinesweeperEnv(**env_config, seed=base_seed + episode_index)
        result = run_episode(env, agent, train=True)
        reward_history.append(result['total_reward'])
        win_history.append(1 if result['won'] else 0)
        step_history.append(result['steps'])

    eval_rewards = []
    eval_wins = []
    eval_steps = []
    rollout = None

    for episode_index in range(eval_episodes):
        env = MinesweeperEnv(**env_config, seed=10_000 + base_seed + episode_index)
        result = run_episode(env, agent, train=False, record=(episode_index == 0))
        eval_rewards.append(result['total_reward'])
        eval_wins.append(1 if result['won'] else 0)
        eval_steps.append(result['steps'])
        if rollout is None:
            rollout = result['rollout']

    if hasattr(agent, 'num_visited_states'):
        visited_states = agent.num_visited_states()
    else:
        visited_states = None

    return {
        'agent_name': agent.name,
        'train_rewards': reward_history,
        'train_wins': win_history,
        'train_steps': step_history,
        'eval_avg_reward': sum(eval_rewards) / len(eval_rewards),
        'eval_win_rate': sum(eval_wins) / len(eval_wins),
        'eval_avg_steps': sum(eval_steps) / len(eval_steps),
        'visited_states': visited_states,
        'rollout': rollout,
    }


def compare_agents(agent_builders, env_config, train_episodes=200, eval_episodes=50):
    results = {}
    for offset, (name, builder) in enumerate(agent_builders.items()):
        agent = builder()
        result = train_agent(
            agent,
            env_config,
            train_episodes=train_episodes,
            eval_episodes=eval_episodes,
            base_seed=1_000 * offset,
        )
        results[name] = result
    return results


def plot_training_curves(results, window=20):
    plt.figure(figsize=(8, 4.5))
    for name, result in results.items():
        curve = moving_average(result['train_rewards'], window)
        plt.plot(curve, label=name)
    plt.xlabel('Episode')
    plt.ylabel(f'Moving average reward (window={window})')
    plt.title('Training reward by agent')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def show_results_table(results):
    lines = [
        '| agent | eval avg reward | eval win rate | eval avg steps | visited states |',
        '|---|---:|---:|---:|---:|',
    ]
    for name, result in results.items():
        visited = '-' if result['visited_states'] is None else str(result['visited_states'])
        lines.append(
            f"| {name} | {result['eval_avg_reward']:.2f} | {result['eval_win_rate']:.2f} | {result['eval_avg_steps']:.2f} | {visited} |"
        )
    display(Markdown('\n'.join(lines)))


def show_rollout(results, agent_name, size):
    rollout = results[agent_name]['rollout']
    if not rollout:
        print('No rollout stored for this agent.')
        return

    columns = len(rollout)
    fig, axes = plt.subplots(1, columns, figsize=(3.0 * columns, 3.0))
    if columns == 1:
        axes = [axes]

    for ax, frame in zip(axes, rollout):
        plot_board(
            frame['state'],
            size=size,
            title=frame['title'],
            exploded_action=frame.get('exploded_action'),
            ax=ax,
        )

    fig.suptitle(f'Sample evaluation rollout: {agent_name}', fontsize=12)
    plt.tight_layout()
    plt.show()

## Starter experiment

The next cell compares:
- a complete random baseline
- the tabular stub as it is now

The default experiment uses `3x3` with `2` mines. Once you implement learning, rerun the comparison. You can also add more agent builders if you want to compare several ideas side by side.

In [ ]:
env_config = {
    'size': 3,
    'num_mines': 2,
    'safe_first_move': True,
    'safe_reward': 1.0,
    'mine_reward': -5.0,
    'win_reward': 10.0,
    'step_penalty': -0.1,
}

agent_builders = {
    'random': lambda: RandomAgent(seed=0, name='random'),
    'tabular_stub': lambda: TabularAgent(alpha=0.2, gamma=0.95, epsilon=0.1, seed=0, name='tabular_stub'),
}

results = compare_agents(
    agent_builders,
    env_config,
    train_episodes=200,
    eval_episodes=50,
)

plot_training_curves(results, window=20)
show_results_table(results)

## Sample rollouts

Use these visualizations to debug behavior, not just final scores. Strange action choices are often easier to spot on a board than in a reward curve.

In [ ]:
show_rollout(results, 'random', size=env_config['size'])
show_rollout(results, 'tabular_stub', size=env_config['size'])

## Your tasks

1. Implement a non-trivial `select_action(...)` method in `TabularAgent`.
2. Implement a tabular update rule in `observe(...)`.
3. Compare your agent against `RandomAgent`.
4. If one idea works, add another agent and compare them.
5. Start from the provided `3x3` with `2` mines, then change the setup only if you have a reason.

Questions worth asking:
- How many states do you visit?
- Is your state representation more detailed than it needs to be?
- How does the rough upper bound of `262,144` visible states affect your design choices?
- What reward choices make learning easier or harder?
- How quickly does the state space become too large for simple tabular methods?
- Do two different update rules behave differently even on tiny boards?